In [ ]:
# ============================================
# LINE -> CHUNK SPLITTING
# Splits each extracted line image into horizontal chunks of
# TARGET_WIDTH +/- TOLERANCE pixels, choosing the actual cut point
# within that window at the column with the LEAST ink — so cuts land in
# the gap between characters instead of through one.
#
# Output naming: <line_image_name>_chunk01.png, _chunk02.png, ...
# ============================================

import cv2
import numpy as np
import os
import glob
from pathlib import Path
from scipy.ndimage import uniform_filter1d

# ============================================
# CONFIG — UPDATE THESE PATHS
# ============================================
line_folder  = r"D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-lines"          # folder of <base>_lineNN.png
chunk_folder = r"D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-line-chunks"    # where chunk crops are saved

TARGET_WIDTH = 200
TOLERANCE    = 30      # actual chunk width will be TARGET_WIDTH +/- TOLERANCE wherever possible

INK_MARGIN     = 20    # a pixel counts as "ink" if it's this much darker than the line's background mode
COL_SMOOTH     = 5     # smoothing window for the column ink-density profile
MIN_LAST_CHUNK = 50    # if the final leftover sliver is narrower than this, merge it into the previous chunk


def ink_column_profile(img):
    """Column-wise ink density. Background here is a flat average color
    (from the reconstruction step) or natural leaf texture — either way,
    actual character strokes are reliably darker than their surroundings,
    so we find the line's own dominant (background) brightness and treat
    anything meaningfully darker than that as ink."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    counts = np.bincount(gray.flatten(), minlength=256)
    bg_val = int(np.argmax(counts))  # the most common gray value = background
    thresh = bg_val - INK_MARGIN
    ink = (gray < thresh).astype(np.uint8)
    col_sum = ink.sum(axis=0).astype(float)
    return uniform_filter1d(col_sum, size=COL_SMOOTH)


def find_chunk_bounds(col_profile, target=TARGET_WIDTH, tol=TOLERANCE, min_last=MIN_LAST_CHUNK):
    """Walk left to right; at each step, look inside [target-tol, target+tol]
    ahead of the current position and cut at whichever column has the least
    ink (ties broken by proximity to the exact target distance)."""
    w = len(col_profile)
    if w <= target + tol:
        return [(0, w)]

    bounds = [0]
    pos = 0
    while True:
        remaining = w - pos
        if remaining <= target + tol:
            bounds.append(w)
            break

        lo = max(pos + target - tol, pos + 1)
        hi = min(pos + target + tol, w - 1)
        window = col_profile[lo:hi + 1]

        min_val = window.min()
        candidates = np.where(window == min_val)[0]
        ideal = (pos + target) - lo
        best = candidates[np.argmin(np.abs(candidates - ideal))]
        cut = lo + int(best)

        if cut <= pos:  # safety net against degenerate windows
            cut = pos + target

        bounds.append(cut)
        pos = cut

    # merge a too-small trailing sliver into the previous chunk
    if len(bounds) >= 3 and (bounds[-1] - bounds[-2]) < min_last:
        bounds.pop(-2)

    return [(bounds[i], bounds[i + 1]) for i in range(len(bounds) - 1)]


def process_line_image(path, out_folder):
    img = cv2.imread(path)
    if img is None:
        print(f"  ❌ Could not read: {path}")
        return 0

    name = Path(path).stem
    profile = ink_column_profile(img)
    chunks = find_chunk_bounds(profile)

    for n, (left, right) in enumerate(chunks, start=1):
        crop = img[:, left:right]
        out_path = os.path.join(out_folder, f"{name}_chunk{n:02d}.png")
        cv2.imwrite(out_path, crop)

    print(f"  ✅ {Path(path).name}: {len(chunks)} chunk(s)")
    return len(chunks)


def main():
    os.makedirs(chunk_folder, exist_ok=True)

    extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff', '*.tif']
    line_paths = []
    for ext in extensions:
        line_paths.extend(glob.glob(os.path.join(line_folder, ext)))
        line_paths.extend(glob.glob(os.path.join(line_folder, ext.upper())))
    line_paths = sorted(set(line_paths))

    if not line_paths:
        print(f"❌ No line images found in: {line_folder}")
        return

    print(f"Found {len(line_paths)} line image(s)")
    print("=" * 60)

    total_chunks = 0
    for p in line_paths:
        total_chunks += process_line_image(p, chunk_folder)

    print("\n" + "=" * 60)
    print("CHUNKING COMPLETE")
    print(f"  Line images processed : {len(line_paths)}")
    print(f"  Total chunks saved    : {total_chunks}")
    print(f"  Outputs saved to      : {chunk_folder}/")


if __name__ == "__main__":
    main()

Found 8824 line image(s)
  ✅ 846_100_line01.png: 5 chunk(s)
  ✅ 846_100_line02.png: 6 chunk(s)
  ✅ 846_100_line03.png: 6 chunk(s)
  ✅ 846_100_line04.png: 6 chunk(s)
  ✅ 846_10_line01.png: 9 chunk(s)
  ✅ 846_10_line02.png: 9 chunk(s)
  ✅ 846_10_line03.png: 9 chunk(s)
  ✅ 846_10_line04.png: 10 chunk(s)
  ✅ 846_10_line05.png: 10 chunk(s)
  ✅ 846_10_line06.png: 10 chunk(s)
  ✅ 846_10_line07.png: 9 chunk(s)
  ✅ 846_11_line01.png: 8 chunk(s)
  ✅ 846_11_line02.png: 7 chunk(s)
  ✅ 846_11_line03.png: 8 chunk(s)
  ✅ 846_11_line04.png: 8 chunk(s)
  ✅ 846_11_line05.png: 8 chunk(s)
  ✅ 846_11_line06.png: 9 chunk(s)
  ✅ 846_11_line07.png: 8 chunk(s)
  ✅ 846_14_line01.png: 8 chunk(s)
  ✅ 846_14_line02.png: 8 chunk(s)
  ✅ 846_14_line03.png: 8 chunk(s)
  ✅ 846_14_line04.png: 8 chunk(s)
  ✅ 846_14_line05.png: 8 chunk(s)
  ✅ 846_14_line06.png: 8 chunk(s)
  ✅ 846_14_line07.png: 8 chunk(s)
  ✅ 846_17_line01.png: 8 chunk(s)
  ✅ 846_17_line02.png: 8 chunk(s)
  ✅ 846_17_line03.png: 8 chunk(s)
  ✅ 846_17_line0